# HDGPSO + HDGPSO-MF walkthrough

This notebook is a companion to the paper. It contains the algorithm code, the multi-fidelity variant, the Demsar (2006) statistical helpers, and a verification step against the saved paper results. Each section is preceded by a short markdown note that explains what is being done so the notebook can be read sequentially.

The reason for a notebook is that the package separates the algorithm into multiple files: `core.py` for HDGPSO, `multifidelity.py` for HDGPSOMF, and `stats.py` for the statistical helpers. This is convenient for using the package as a library, but it is less convenient when the goal is to follow along with the paper, modify a stage, or debug an issue. The notebook keeps the entire pipeline in one runnable document.

## Contents

1. Background on the HPO problem, followed by short summaries of DE, GWO, and PSO.
2. The motivation for combining these three operators, with reference to the No Free Lunch theorem and the strengths and weaknesses of each method.
3. The search space primitives and the optimizer call.
4. A small demo on a synthetic function and a sklearn task.
5. The multi-fidelity wrapper.
6. The Demsar (2006) battery applied to a small synthetic benchmark.
7. Loading the saved paper benchmark CSV and verifying the headline results.


---
## 2. Setup

### 1.5 Why these three operators together?

Each of the three algorithms has its own strengths and weaknesses. The combination as sequential stages inside one iteration is intended to keep the useful properties of each method and reduce the effect of the situations in which each method alone would stall.

Two well-known results in optimization theory motivate the design.

The first is the No Free Lunch theorem (Wolpert & Macready, 1997). The theorem states that, averaged over all possible objective functions, no optimizer is better than any other. The practical implication is narrower than the formal statement: when the objective function is not known in advance, it is safer to combine several search behaviors that fail in different ways than to commit to a single one. DE explores the space using population differences. GWO concentrates around the current top three candidates. PSO refines positions using memory of past good positions. These three failure modes do not overlap.

The second motivation is about the type of the search space. Metaheuristics like DE, GWO, and PSO update real-valued coordinates, and the search space decoder then converts those coordinates into the correct hyperparameter type (integer, float, or categorical) by clamping or rounding. These methods do not assume the loss surface is smooth in any specific direction. Surrogate methods such as Gaussian-Process Bayesian Optimization tend to perform best when smoothness assumptions hold. They are less effective on search spaces that are dominated by discrete and categorical dimensions, because the Gaussian Process cannot place a clean kernel on those dimensions.

These two observations together produce a working hypothesis: a metaheuristic stack should be especially competitive on HPO problems that are dominated by integer and categorical hyperparameters, such as tree-ensemble models. This is the pattern that the benchmark in the paper shows, where HDGPSO wins all three Gradient-Boosted-tree cells outright.

#### What each operator brings

Differential Evolution. For each candidate, three other population members are selected at random, and their difference is scaled by `F` and added back. The magnitude of the perturbation adapts to the current spread of the population. A widely spread population produces large perturbations, and an already concentrated one produces small ones. This is a useful property, because no manual cooling schedule on the step size is required. The limitation of DE is that it does not retain per-member memory of past good positions. Once a candidate is replaced by a better trial, the previous position is lost.

Grey Wolf Optimizer. The top three members of the population (α, β, δ) act as leaders. Every other member is updated toward a weighted blend of these three leaders, with the strength of the pull decreasing as iterations progress. This produces a clear exploitation pressure toward the part of the space that currently looks promising. The trade-off is that information outside the top three is not tracked.

Particle Swarm Optimization. Each particle stores its own best-ever position (`pbest`) and the swarm stores a global best (`gbest`). The velocity update combines the current velocity with a pull toward `pbest` and a pull toward `gbest`. The inertia weight on the current velocity decays linearly across iterations so the swarm starts broad and becomes tighter over time. PSO is effective for fine refinement once the search is approximately in the correct region. The limitation is that if diversity is lost too early, the swarm may converge prematurely.

By running DE, then GWO, then PSO in every iteration, every member of the population is pulled by a different mechanism three times. When one mechanism's schedule has decayed, another remains active.

#### The role of the surrogate

A RandomForest surrogate is fit on the trial history every four iterations once at least twelve points are available. The surrogate is not used to replace any objective evaluation. It is used only to score proposed candidates between the DE, GWO, and PSO stages. For each proposed candidate, several perturbed neighbors are generated, their mean prediction and tree-variance are computed, and the candidate with the best lower confidence bound score is kept. This step screens out clearly weak proposals before the real objective is invoked.

The choice of RandomForest, rather than a Gaussian Process, keeps the dependency footprint small. Scikit-learn already provides everything needed. RandomForest also handles mixed-type spaces naturally, while a Gaussian Process requires additional kernel design to do so.


---
## 3. Search space primitives

HDGPSO operates on float vectors internally — every hyperparameter is encoded into a single coordinate that the algorithm manipulates. The `Dimension` subclasses know how to *decode* a coordinate back to a typed value (`int`, `float`, or a `Categorical` choice).

- **`Float`** — continuous values, optionally log-scaled (use `log=True` when the range spans several orders of magnitude, e.g. learning rates from `1e-5` to `1e-1`).
- **`Int`** — integer values; internally stored on a `[low - 0.5, high + 0.5]` real interval and rounded on decode.
- **`Categorical`** — unordered choices; internally stored as an index that gets rounded to the nearest valid choice.

All three live on bounded internal coordinates so DE/GWO/PSO operators can move freely without worrying about type constraints.

In [1]:
# Standard imports used throughout
from __future__ import annotations

import math
import time
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as sps

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
np.random.seed(0)

---
## 2. Search space primitives

HDGPSO operates on float vectors internally — every hyperparameter is encoded into a single coordinate that the algorithm manipulates. The `Dimension` subclasses know how to *decode* a coordinate back to a typed value (`int`, `float`, or a `Categorical` choice).

- **`Float`** — continuous values, optionally log-scaled (use `log=True` when the range spans several orders of magnitude, e.g. learning rates from `1e-5` to `1e-1`).
- **`Int`** — integer values; internally stored on a `[low - 0.5, high + 0.5]` real interval and rounded on decode.
- **`Categorical`** — unordered choices; internally stored as an index that gets rounded to the nearest valid choice.

All three live on bounded internal coordinates so DE/GWO/PSO operators can move freely without worrying about type constraints.

---
## 4. `OptimizeResult` — what every optimizer returns

A simple dataclass that bundles the optimization output: best parameters found, best loss, full trial history (a pandas DataFrame), number of evaluations, elapsed wall-clock, and why we stopped (iterations / budget / time / early_stop).

In [5]:
# Test the search space: build one, sample, decode
from hdgpso import Categorical, Float, Int, SearchSpace
space = SearchSpace({
    "lr":         Float(1e-5, 1e-1, log=True),
    "dropout":    Float(0.0, 0.5),
    "hidden_dim": Int(16, 256),
    "activation": Categorical(["relu", "gelu", "tanh"]),
})
rng = np.random.default_rng(42)
samples = space.sample(rng, n=5)
print(f"Internal coordinates shape: {samples.shape}\n")
for i, x in enumerate(samples):
    print(f"Sample {i}: {space.decode(x)}")

Internal coordinates shape: (5, 4)

Sample 0: {'lr': 0.012468786659075646, 'dropout': 0.21943921987602616, 'hidden_dim': 222, 'activation': 'tanh'}
Sample 1: {'lr': 2.3807258718953526e-05, 'dropout': 0.48781117581837796, 'hidden_dim': 199, 'activation': 'tanh'}
Sample 2: {'lr': 3.254277104330505e-05, 'dropout': 0.22519296894778357, 'hidden_dim': 105, 'activation': 'tanh'}
Sample 3: {'lr': 0.003762361141943257, 'dropout': 0.411380806635415, 'hidden_dim': 122, 'activation': 'relu'}
Sample 4: {'lr': 0.0016532523475753852, 'dropout': 0.03190862805208766, 'hidden_dim': 215, 'activation': 'gelu'}


---
## 5. The HDGPSO three-stage algorithm

The algorithm described below is the same as the one detailed in Section 1. Each iteration applies three operator stages on the population, followed by a personal-best and global-best update.

### Stage 1: Differential Evolution (DE/rand/1 with binary crossover)

For each member `i` in the population, three other distinct members `a`, `b`, and `c` are selected. A donor vector is formed as `v = x_a + F * (x_b - x_c)`. The donor is then combined with the current `x_i` through binary crossover with rate `CR`, producing the trial `u`. The trial is evaluated, and it replaces `x_i` only if it improves the loss. The default values used in this implementation are `F = 0.8` and `CR = 0.5`.

### Stage 2: Grey Wolf Optimizer

The population is sorted by current loss, and the top three members are labeled α, β, and δ. For every other member, the position update follows the Mirjalili (2014) rule and moves the member toward a weighted blend of the three leaders. The strength of the pull decreases across iterations as the coefficient `a` decays from 2 to 0.

### Stage 3: Particle Swarm Optimization

Each particle stores a personal-best position and a per-particle velocity, and the swarm stores a global best. The velocity is updated as

```
v_i = w * v_i + c1 * r1 * (pbest_i - x_i) + c2 * r2 * (gbest - x_i)
```

with `c1 = c2 = 2.0` and `w` decaying linearly from 0.7 to 0.4 across iterations. The new position is `x_i = x_i + v_i`.

### Between stages: the surrogate filter

After each stage proposes new positions, those positions pass through the RandomForest filter described above before being evaluated by the real objective.


In [6]:
@dataclass
class OptimizeResult:
    best_params: Dict[str, Any]
    best_loss: float
    history: pd.DataFrame
    n_evals: int
    elapsed_seconds: float
    stopped_reason: str = "iterations"

    def __repr__(self) -> str:
        return (
            f"OptimizeResult(best_loss={self.best_loss:.6g}, "
            f"n_evals={self.n_evals}, elapsed={self.elapsed_seconds:.1f}s, "
            f"stopped={self.stopped_reason})"
        )

---
## 6. Demo: minimize the sphere function

A trivial 3D minimization to confirm the algorithm converges correctly. The sphere function `f(x, y, z) = x² + y² + z²` has its global optimum at the origin with value 0.

In [8]:
class HDGPSO:
    """Hybrid DE-GWO-PSO optimizer with optional RandomForest surrogate."""
    name = "HDGPSO"

    def __init__(
        self,
        space, objective,
        population_size=10, iterations=20,
        F=0.8, CR=0.5, c1=2.0, c2=2.0, w_max=0.7, w_min=0.4,
        early_stop_patience=None, time_budget_seconds=None, eval_budget=None,
        use_surrogate=True, surrogate_pool=16, surrogate_refit_every=4,
        surrogate_min_history=12, surrogate_kappa=0.0,
        restart_patience=None, restart_fraction=0.5,
        seed=None, verbose=False,
    ):
        if not isinstance(space, SearchSpace):
            space = SearchSpace(space)
        if int(population_size) < 4:
            raise ValueError(f"population_size must be >= 4; got {population_size}")
        self.space = space; self.objective = objective
        self.population_size = int(population_size); self.iterations = int(iterations)
        self.F, self.CR = float(F), float(CR)
        self.c1, self.c2 = float(c1), float(c2)
        self.w_max, self.w_min = float(w_max), float(w_min)
        self.early_stop_patience = early_stop_patience
        self.time_budget_seconds = time_budget_seconds
        self.eval_budget = eval_budget
        self.use_surrogate = bool(use_surrogate)
        self.surrogate_pool = int(surrogate_pool)
        self.surrogate_refit_every = int(surrogate_refit_every)
        self.surrogate_min_history = int(surrogate_min_history)
        self.surrogate_kappa = float(surrogate_kappa)
        self.restart_patience = restart_patience
        self.restart_fraction = float(restart_fraction)
        self.rng = np.random.default_rng(seed); self.verbose = bool(verbose)
        self._surrogate = None; self._last_surrogate_refit_at = -1
        self._stagnation_iters = 0; self._best_loss_at_last_check = float("inf")
        self.population = None; self._latest_losses = None
        self._velocities = None; self._pbest = None; self._pbest_losses = None
        self.best_x = None; self.best_loss = float("inf")
        self._history: List[Dict[str, Any]] = []; self._t0 = 0.0

    # ---- evaluation + budget accounting ---------------------------------
    def _eval(self, x, iteration):
        params = self.space.decode(x)
        try:
            loss = float(self.objective(params))
        except Exception as exc:
            loss = float("inf")
            params = {**params, "_error": repr(exc)}
        if not np.isfinite(loss):
            loss = float("inf")
        self._history.append({
            "iteration": iteration, "loss": loss,
            "elapsed": time.time() - self._t0,
            "optimizer": self.name, **params,
        })
        if loss < self.best_loss:
            self.best_loss = loss; self.best_x = x.copy()
        return loss

    def _budget_exhausted(self):
        if self.eval_budget is not None and len(self._history) >= self.eval_budget:
            return True
        if self.time_budget_seconds is not None:
            if (time.time() - self._t0) >= self.time_budget_seconds:
                return True
        return False

    # ---- surrogate model --------------------------------------------------
    def _refit_surrogate(self, iteration):
        if not self.use_surrogate or len(self._history) < self.surrogate_min_history:
            return
        if iteration == self._last_surrogate_refit_at: return
        if (iteration - self._last_surrogate_refit_at) < self.surrogate_refit_every: return
        try:
            from sklearn.ensemble import RandomForestRegressor
        except ImportError:
            self.use_surrogate = False; return
        X, y = [], []
        for rec in self._history:
            if not np.isfinite(rec.get("loss", float("inf"))): continue
            try:
                vec = [self.space.dims[n].to_internal(rec[n]) for n in self.space.names]
            except Exception:
                continue
            X.append(vec); y.append(rec["loss"])
        if len(X) < self.surrogate_min_history: return
        X_arr, y_arr = np.asarray(X), np.asarray(y)
        y_arr = np.minimum(y_arr, np.quantile(y_arr, 0.99))
        self._surrogate = RandomForestRegressor(
            n_estimators=30, max_depth=10, n_jobs=1, random_state=42
        ).fit(X_arr, y_arr)
        self._last_surrogate_refit_at = iteration

    def _surrogate_predict_with_std(self, X):
        per_tree = np.stack([t.predict(X) for t in self._surrogate.estimators_])
        return per_tree.mean(axis=0), per_tree.std(axis=0)

    def _surrogate_filter(self, base):
        if self._surrogate is None or not self.use_surrogate:
            return base
        K = max(self.surrogate_pool, 2)
        scale = 0.1 * (self.space.highs - self.space.lows)
        noise = self.rng.normal(0.0, 1.0, size=(K - 1, self.space.n_dims)) * scale
        candidates = np.vstack([base.reshape(1, -1), base + noise])
        candidates = self.space.clip(candidates)
        try:
            mean, std = self._surrogate_predict_with_std(candidates)
            acq = mean - self.surrogate_kappa * std   # LCB for minimization
            return candidates[int(np.argmin(acq))]
        except Exception:
            return base

    # ---- stagnation-triggered restart ------------------------------------
    def _maybe_restart(self):
        if self.restart_patience is None: return False
        if self.best_loss + 1e-12 < self._best_loss_at_last_check:
            self._best_loss_at_last_check = self.best_loss
            self._stagnation_iters = 0; return False
        self._stagnation_iters += 1
        if self._stagnation_iters < self.restart_patience: return False
        n_restart = max(1, int(self.restart_fraction * self.population_size))
        worst = np.argsort(self._latest_losses)[-n_restart:]
        self.population[worst] = self.space.sample(self.rng, n_restart)
        if self._pbest is not None:
            self._pbest[worst] = self.population[worst]
        if self._velocities is not None:
            self._velocities[worst] = 0.0
        self._stagnation_iters = 0; return True

    # ---- GWO leadership step ---------------------------------------------
    def _gwo_step(self, iteration):
        n, d = self.population.shape
        order = np.argsort(self._latest_losses)
        alpha, beta = self.population[order[0]], self.population[order[1 if n > 1 else 0]]
        delta = self.population[order[2 if n > 2 else (1 if n > 1 else 0)]]
        a = 2.0 * (1.0 - iteration / max(self.iterations, 1))

        def leader_step(leader):
            r1 = self.rng.random((n, d)); r2 = self.rng.random((n, d))
            A = 2.0 * a * r1 - a; C = 2.0 * r2
            return leader - A * np.abs(C * leader - self.population)

        self.population = (leader_step(alpha) + leader_step(beta) + leader_step(delta)) / 3.0

    # ---- PSO update -------------------------------------------------------
    def _update_pbest(self):
        better = self._latest_losses < self._pbest_losses
        if better.any():
            self._pbest[better] = self.population[better]
            self._pbest_losses[better] = self._latest_losses[better]

    def _pso_step(self, iteration):
        n, d = self.population.shape
        frac = iteration / max(self.iterations, 1)
        w = self.w_max - (self.w_max - self.w_min) * frac
        r1 = self.rng.random((n, d)); r2 = self.rng.random((n, d))
        cognitive = self.c1 * r1 * (self._pbest - self.population)
        social = self.c2 * r2 * (self.best_x - self.population)
        self._velocities = w * self._velocities + cognitive + social
        self.population = self.population + self._velocities

    # ---- main optimization loop -------------------------------------------
    def optimize(self):
        self._t0 = time.time(); self._history.clear()
        self.population = self.space.sample(self.rng, self.population_size)
        self.best_x = None; self.best_loss = float("inf")
        self._latest_losses = np.array(
            [self._eval(self.population[i], 0) for i in range(self.population_size)]
        )
        self._pbest = self.population.copy()
        self._pbest_losses = self._latest_losses.copy()
        self._velocities = np.zeros_like(self.population)
        stagnation, prev_best, stopped = 0, self.best_loss, "iterations"

        for it in range(1, self.iterations + 1):
            if self._budget_exhausted(): stopped = "budget"; break
            self._refit_surrogate(it)

            # Stage 1: DE -----------------------------------------------------
            new_pop, new_losses = self.population.copy(), self._latest_losses.copy()
            for i in range(self.population_size):
                idxs = [j for j in range(self.population_size) if j != i]
                a_i, b_i, c_i = self.rng.choice(idxs, 3, replace=False)
                donor = self.population[a_i] + self.F * (self.population[b_i] - self.population[c_i])
                mask = self.rng.random(self.space.n_dims) < self.CR
                if not mask.any(): mask[self.rng.integers(self.space.n_dims)] = True
                trial = self.space.clip(np.where(mask, donor, self.population[i]))
                trial = self._surrogate_filter(trial)
                tl = self._eval(trial, it)
                if tl < self._latest_losses[i]:
                    new_pop[i] = trial; new_losses[i] = tl
                if self._budget_exhausted(): stopped = "budget"; break
            self.population, self._latest_losses = new_pop, new_losses
            self._update_pbest()
            if stopped == "budget": break

            # Stage 2: GWO ----------------------------------------------------
            self._gwo_step(it)
            self.population = self.space.clip(self.population)
            if self.use_surrogate and self._surrogate is not None:
                self.population = np.stack(
                    [self._surrogate_filter(self.population[i])
                     for i in range(self.population_size)]
                )
                self.population = self.space.clip(self.population)
            new_losses = np.empty(self.population_size)
            for i in range(self.population_size):
                if self._budget_exhausted():
                    new_losses[i:] = self._latest_losses[i:]; stopped = "budget"; break
                new_losses[i] = self._eval(self.population[i], it)
            self._latest_losses = new_losses; self._update_pbest()
            if stopped == "budget": break

            # Stage 3: PSO ----------------------------------------------------
            self._pso_step(it)
            self.population = self.space.clip(self.population)
            if self.use_surrogate and self._surrogate is not None:
                self.population = np.stack(
                    [self._surrogate_filter(self.population[i])
                     for i in range(self.population_size)]
                )
                self.population = self.space.clip(self.population)
            new_losses = np.empty(self.population_size)
            for i in range(self.population_size):
                if self._budget_exhausted():
                    new_losses[i:] = self._latest_losses[i:]; stopped = "budget"; break
                new_losses[i] = self._eval(self.population[i], it)
            self._latest_losses = new_losses; self._update_pbest()
            if stopped == "budget": break

            # Optional restart-on-stagnation ----------------------------------
            if self._maybe_restart():
                for i in range(self.population_size):
                    if self._budget_exhausted(): stopped = "budget"; break
                    self._latest_losses[i] = self._eval(self.population[i], it)
                self._update_pbest()
                if stopped == "budget": break

            if self.verbose:
                print(f"[{self.name}] iter {it}/{self.iterations} best={self.best_loss:.6g}")
            if self.early_stop_patience is not None:
                if self.best_loss < prev_best - 1e-12:
                    stagnation = 0; prev_best = self.best_loss
                else:
                    stagnation += 1
                    if stagnation >= self.early_stop_patience:
                        stopped = "early_stop"; break

        elapsed = time.time() - self._t0
        hist = pd.DataFrame(self._history)
        if not hist.empty:
            hist["running_best"] = hist["loss"].cummin()
        return OptimizeResult(
            best_params=self.space.decode(self.best_x) if self.best_x is not None else {},
            best_loss=self.best_loss, history=hist, n_evals=len(self._history),
            elapsed_seconds=elapsed, stopped_reason=stopped,
        )

---
## 5. Demo: minimize the sphere function

A trivial 3D minimization to confirm the algorithm converges correctly. The sphere function `f(x, y, z) = x² + y² + z²` has its global optimum at the origin with value 0.

---
## 7. Realistic example — RandomForest on Breast Cancer

Tune a `RandomForestClassifier` (5 mixed-type hyperparameters) by minimizing **negative cross-validated accuracy**.

In [9]:
# Convergence plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(result.history['running_best'], 'r-', linewidth=2)
ax.set_xlabel('Evaluation #'); ax.set_ylabel('Best loss so far (log scale)')
ax.set_title('HDGPSO convergence on sphere function')
ax.grid(alpha=0.3); plt.show()

NameError: name 'result' is not defined

---
## 8. `HDGPSOMF` — multi-fidelity extension

This implements the BOHB-style successive halving described in §1.7. When each objective evaluation is expensive, HDGPSOMF probes candidates cheaply at `low_fidelity` first and only verifies the top fraction at full fidelity.

1. Each DE/GWO/PSO stage proposes `population_size` candidates as before.
2. **All proposals are first evaluated at `low_fidelity` (cheap probe)** — e.g., training with 30% of the configured epochs.
3. The top `verify_fraction` (by low-fidelity loss) are **re-evaluated at full fidelity** for verification.
4. Surrogate is trained **only on full-fidelity points** to avoid noisy low-fidelity miscalibration.

Budget is measured in **fidelity-units**: one full-fidelity eval costs 1.0 unit; a probe at `fidelity=0.3` costs 0.3 units. At the same nominal budget, HDGPSOMF can run 2-3× more candidate proposals — better exploration when each eval is expensive.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

X, y = load_breast_cancer(return_X_y=True)

rf_space = SearchSpace({
    "n_estimators":      Int(20, 300),
    "max_depth":         Int(2, 20),
    "min_samples_split": Int(2, 20),
    "min_samples_leaf":  Int(1, 20),
    "max_features":      Categorical(["sqrt", "log2", 0.5, 1.0]),
})

def rf_objective(params):
    model = RandomForestClassifier(**params, random_state=0, n_jobs=1)
    return -cross_val_score(model, X, y, cv=3).mean()  # lower is better

rf_result = HDGPSO(
    rf_space, rf_objective,
    population_size=6, iterations=5, seed=0,
).optimize()
print(f"Best params: {rf_result.best_params}")
print(f"Best CV acc: {-rf_result.best_loss:.4f}")
print(f"Evaluations: {rf_result.n_evals}")

---
## 9. Demo — noisy Rosenbrock with multi-fidelity

A synthetic 2D objective where the *full-fidelity* evaluation is slow (`time.sleep(0.05)`) and the *low-fidelity* evaluation is fast but noisy. HDGPSOMF should make more candidate proposals at the same budget.

In [ ]:
class HDGPSOMF(HDGPSO):
    """HDGPSO with multi-fidelity successive halving."""
    name = "HDGPSO-MF"

    def __init__(
        self,
        space, objective,
        population_size=5, iterations=30,
        F=0.8, CR=0.5, c1=2.0, c2=2.0, w_max=0.7, w_min=0.4,
        eval_budget=None, low_fidelity=0.3, verify_fraction=0.4,
        use_surrogate=True, surrogate_pool=16, surrogate_refit_every=4,
        surrogate_min_history=6, surrogate_kappa=0.0,
        restart_patience=None, seed=None, verbose=False,
    ):
        super().__init__(
            space=space, objective=objective,
            population_size=population_size, iterations=iterations,
            F=F, CR=CR, c1=c1, c2=c2, w_max=w_max, w_min=w_min,
            eval_budget=None,   # we manage budget in fidelity-units below
            use_surrogate=use_surrogate, surrogate_pool=surrogate_pool,
            surrogate_refit_every=surrogate_refit_every,
            surrogate_min_history=surrogate_min_history,
            surrogate_kappa=surrogate_kappa,
            restart_patience=restart_patience,
            seed=seed, verbose=verbose,
        )
        self.fidelity_budget = float(eval_budget) if eval_budget else None
        self.low_fidelity = float(low_fidelity)
        self.verify_fraction = float(verify_fraction)
        self._fidelity_used = 0.0

    def _eval(self, x, iteration, fidelity=1.0):
        params = self.space.decode(x)
        try:
            loss = float(self.objective(params, fidelity=fidelity))
        except TypeError:
            loss = float(self.objective(params))
        if not np.isfinite(loss):
            loss = float("inf")
        self._fidelity_used += fidelity
        self._history.append({
            "iteration": iteration, "loss": loss, "fidelity": fidelity,
            "elapsed": time.time() - self._t0, "optimizer": self.name, **params,
        })
        if fidelity >= 0.99 and loss < self.best_loss:
            self.best_loss = loss; self.best_x = x.copy()
        return loss

    def _budget_exhausted(self):
        if self.fidelity_budget is not None:
            return self._fidelity_used >= self.fidelity_budget
        return False

    def _verify_top(self, candidates, low_losses, iteration):
        n = len(candidates)
        n_verify = max(1, int(np.ceil(self.verify_fraction * n)))
        top_idx = np.argsort(low_losses)[:n_verify]
        full_losses = low_losses.copy()
        for i in top_idx:
            if self._budget_exhausted(): break
            full_losses[i] = self._eval(candidates[i], iteration, fidelity=1.0)
        return full_losses

    def _refit_surrogate(self, iteration):
        # Train surrogate only on FULL-fidelity points (clean signal).
        if not self.use_surrogate: return
        full_hist = [h for h in self._history
                     if h.get("fidelity", 1.0) >= 0.99
                     and np.isfinite(h.get("loss", float("inf")))]
        if len(full_hist) < self.surrogate_min_history: return
        if iteration == self._last_surrogate_refit_at: return
        if (iteration - self._last_surrogate_refit_at) < self.surrogate_refit_every: return
        try:
            from sklearn.ensemble import RandomForestRegressor
        except ImportError:
            self.use_surrogate = False; return
        X, y = [], []
        for rec in full_hist:
            try:
                vec = [self.space.dims[n].to_internal(rec[n]) for n in self.space.names]
            except Exception:
                continue
            X.append(vec); y.append(rec["loss"])
        if len(X) < self.surrogate_min_history: return
        X_arr, y_arr = np.asarray(X), np.asarray(y)
        y_arr = np.minimum(y_arr, np.quantile(y_arr, 0.99))
        self._surrogate = RandomForestRegressor(
            n_estimators=30, max_depth=10, n_jobs=1, random_state=42
        ).fit(X_arr, y_arr)
        self._last_surrogate_refit_at = iteration

    def optimize(self):
        self._t0 = time.time(); self._history.clear(); self._fidelity_used = 0.0
        self.population = self.space.sample(self.rng, self.population_size)
        self.best_x = None; self.best_loss = float("inf")
        # Initial pop: evaluate at FULL fidelity for clean surrogate baseline
        self._latest_losses = np.empty(self.population_size)
        for i in range(self.population_size):
            if self._budget_exhausted(): break
            self._latest_losses[i] = self._eval(self.population[i], 0, fidelity=1.0)
        self._pbest = self.population.copy()
        self._pbest_losses = self._latest_losses.copy()
        self._velocities = np.zeros_like(self.population)
        stopped = "iterations"

        for it in range(1, self.iterations + 1):
            if self._budget_exhausted(): stopped = "budget"; break
            self._refit_surrogate(it)

            # Stage 1: DE at LOW fidelity, verify top --------------------------
            trials = np.empty_like(self.population)
            for i in range(self.population_size):
                idxs = [j for j in range(self.population_size) if j != i]
                a_i, b_i, c_i = self.rng.choice(idxs, 3, replace=False)
                donor = self.population[a_i] + self.F * (self.population[b_i] - self.population[c_i])
                mask = self.rng.random(self.space.n_dims) < self.CR
                if not mask.any(): mask[self.rng.integers(self.space.n_dims)] = True
                trial = self.space.clip(np.where(mask, donor, self.population[i]))
                trials[i] = self._surrogate_filter(trial)
            low_losses = np.empty(self.population_size)
            for i in range(self.population_size):
                if self._budget_exhausted(): low_losses[i] = float("inf")
                else: low_losses[i] = self._eval(trials[i], it, fidelity=self.low_fidelity)
            full_losses = self._verify_top(trials, low_losses, it)
            for i in range(self.population_size):
                if full_losses[i] < self._latest_losses[i]:
                    self.population[i] = trials[i]; self._latest_losses[i] = full_losses[i]
            self._update_pbest()
            if self._budget_exhausted(): stopped = "budget"; break

            # Stage 2: GWO at LOW fidelity, verify top -------------------------
            self._gwo_step(it); self.population = self.space.clip(self.population)
            if self.use_surrogate and self._surrogate is not None:
                self.population = np.stack([self._surrogate_filter(self.population[i])
                                            for i in range(self.population_size)])
                self.population = self.space.clip(self.population)
            low_losses = np.empty(self.population_size)
            for i in range(self.population_size):
                if self._budget_exhausted(): low_losses[i] = float("inf")
                else: low_losses[i] = self._eval(self.population[i], it, fidelity=self.low_fidelity)
            self._latest_losses = self._verify_top(self.population, low_losses, it)
            self._update_pbest()
            if self._budget_exhausted(): stopped = "budget"; break

            # Stage 3: PSO at LOW fidelity, verify top -------------------------
            self._pso_step(it); self.population = self.space.clip(self.population)
            if self.use_surrogate and self._surrogate is not None:
                self.population = np.stack([self._surrogate_filter(self.population[i])
                                            for i in range(self.population_size)])
                self.population = self.space.clip(self.population)
            low_losses = np.empty(self.population_size)
            for i in range(self.population_size):
                if self._budget_exhausted(): low_losses[i] = float("inf")
                else: low_losses[i] = self._eval(self.population[i], it, fidelity=self.low_fidelity)
            self._latest_losses = self._verify_top(self.population, low_losses, it)
            self._update_pbest()

            if self.verbose:
                print(f"[{self.name}] iter {it} best={self.best_loss:.6g} "
                      f"used={self._fidelity_used:.1f}/{self.fidelity_budget}")

        elapsed = time.time() - self._t0
        hist = pd.DataFrame(self._history)
        if not hist.empty:
            hist["running_best"] = hist.apply(
                lambda r: r["loss"] if r.get("fidelity", 1.0) >= 0.99 else np.nan,
                axis=1
            ).cummin().ffill()
        return OptimizeResult(
            best_params=self.space.decode(self.best_x) if self.best_x is not None else {},
            best_loss=self.best_loss, history=hist, n_evals=len(self._history),
            elapsed_seconds=elapsed, stopped_reason=stopped,
        )

---
## 10. Statistical test details — Demsar (2006)

When we compare N tuners across K (dataset, model, seed) cells, we have to control the family-wise error rate. The standard methodology is Demsar (2006):

1. **Friedman test** — global null "all tuners equivalent". Reject before any pairwise claim.
2. **Nemenyi post-hoc** with Studentized-range critical difference (CD). Two tuners differ significantly if their mean ranks differ by more than CD.
3. **Critical Difference (CD) diagram** — visual summary.
4. **Cliff's δ** — non-parametric effect size in [-1, 1].
5. **Bootstrap 95% CI** on per-tuner mean rank.

These are implemented inline below.

In [ ]:
import time as _time

def rosenbrock_mf(params, fidelity=1.0):
    _time.sleep(0.05 * fidelity)
    x, y = params["x"], params["y"]
    truth = (1 - x) ** 2 + 100 * (y - x * x) ** 2
    # Inject noise that vanishes as fidelity → 1
    noise = np.random.default_rng(
        int((x * 1e3 + y * 1e2) * 1e6) % (2**31)
    ).normal(0, (1.0 - fidelity) * 0.5)
    return truth + noise

rosen_space = SearchSpace({"x": Float(-2, 2), "y": Float(-2, 2)})

# Compare HDGPSO and HDGPSOMF at the same nominal budget
print("HDGPSO (single-fidelity, budget=20):")
r1 = HDGPSO(rosen_space,
            lambda p: rosenbrock_mf(p, fidelity=1.0),
            population_size=4, iterations=4, seed=0).optimize()
print(f"  best_loss = {r1.best_loss:.4f}, n_evals = {r1.n_evals}\n")

print("HDGPSOMF (low_fidelity=0.3, eval_budget=20 fidelity-units):")
r2 = HDGPSOMF(rosen_space, rosenbrock_mf,
              eval_budget=20, low_fidelity=0.3, verify_fraction=0.4,
              population_size=4, seed=0).optimize()
print(f"  best_loss = {r2.best_loss:.4f}, n_evals = {r2.n_evals}")
print(f"  - low-fidelity probes: {(r2.history['fidelity'] < 0.99).sum()}")
print(f"  - full-fidelity evals: {(r2.history['fidelity'] >= 0.99).sum()}")

---
## 11. Demo — synthetic 8-tuner benchmark

Build a small synthetic dataset where we *know* HDGPSO is the best tuner, then run the full Demsar battery on it.

In [ ]:
# Nemenyi critical-value table (Demsar 2006, alpha=0.05, two-tailed)
_Q_ALPHA_05 = {
    2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949,
    8: 3.031, 9: 3.102, 10: 3.164,
}

def critical_difference(k: int, n_datasets: int, alpha: float = 0.05) -> float:
    """Nemenyi CD: rank gaps less than this are NOT significantly different."""
    q = _Q_ALPHA_05[k]
    return q * np.sqrt(k * (k + 1) / (6.0 * n_datasets))


def build_rank_matrix(summary_df: pd.DataFrame) -> pd.DataFrame:
    """One row per (dataset, model, seed); columns = tuners; cells = ranks."""
    pivot = summary_df.pivot_table(
        index=["dataset", "model", "seed"], columns="tuner",
        values="best_loss", aggfunc="first",
    ).dropna()
    return pivot.rank(axis=1, method="average")


def friedman_test(summary_df, alpha=0.05):
    """Friedman omnibus test that all tuners have equal expected rank."""
    ranks = build_rank_matrix(summary_df)
    cols = [ranks[c].values for c in ranks.columns]
    stat, p = sps.friedmanchisquare(*cols)
    return {
        "chi2": float(stat), "pvalue": float(p),
        "n_datasets": len(ranks), "n_tuners": ranks.shape[1],
        "reject_null": p < alpha,
    }


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Cliff's δ effect size in [-1, 1]. Positive δ means x tends to be lower."""
    x, y = np.asarray(x), np.asarray(y)
    diffs = x[:, None] - y[None, :]
    return float((np.sum(diffs < 0) - np.sum(diffs > 0)) / (len(x) * len(y)))


def hdgpso_vs_baselines_table(summary_df, target="HDGPSO", alpha=0.05):
    """Per-baseline: rank delta, Wilcoxon p, Cliff's δ, Nemenyi-significance."""
    ranks = build_rank_matrix(summary_df)
    mean_ranks = ranks.mean(axis=0)
    n = len(ranks); k = ranks.shape[1]
    cd = critical_difference(k, n, alpha)
    pivot = summary_df.pivot_table(
        index=["dataset", "model", "seed"], columns="tuner",
        values="best_loss", aggfunc="first",
    ).dropna()
    rows = []
    target_losses = pivot[target].values
    for other in mean_ranks.index:
        if other == target: continue
        other_losses = pivot[other].values
        try:
            _, wp = sps.wilcoxon(target_losses, other_losses)
        except ValueError:
            wp = np.nan
        rank_delta = float(mean_ranks[other] - mean_ranks[target])
        rows.append({
            "baseline": other,
            "mean_rank_target": float(mean_ranks[target]),
            "mean_rank_baseline": float(mean_ranks[other]),
            "rank_delta": rank_delta,
            "cliffs_delta": cliffs_delta(target_losses, other_losses),
            "wilcoxon_p": float(wp) if wp == wp else float("nan"),
            "nemenyi_significant": rank_delta > cd,
            "CD_at_alpha": cd,
        })
    return pd.DataFrame(rows).sort_values("rank_delta", ascending=False).reset_index(drop=True)


def bootstrap_rank_ci(summary_df, n_boot=2000, alpha=0.05, seed=0):
    """Bootstrap 95% CI on per-tuner mean rank."""
    ranks = build_rank_matrix(summary_df)
    rng = np.random.default_rng(seed)
    n = len(ranks)
    boot_means = np.empty((n_boot, ranks.shape[1]))
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot_means[b] = ranks.iloc[idx].mean(axis=0).values
    lo = np.quantile(boot_means, alpha / 2, axis=0)
    hi = np.quantile(boot_means, 1 - alpha / 2, axis=0)
    return pd.DataFrame({
        "tuner": ranks.columns, "mean_rank": ranks.mean(axis=0).values,
        "ci_lo_95": lo, "ci_hi_95": hi,
    }).sort_values("mean_rank").reset_index(drop=True)


def cd_diagram(summary_df, alpha=0.05, ax=None, title=None):
    """Render a Critical Difference (Demsar 2006) diagram."""
    ranks = build_rank_matrix(summary_df)
    mean_ranks = ranks.mean(axis=0).sort_values()
    tuners = list(mean_ranks.index); k = len(tuners); n = len(ranks)
    cd = critical_difference(k, n, alpha)

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 2.5 + 0.25 * k))
    min_r = float(np.floor(mean_ranks.min())) - 0.2 * (mean_ranks.max() - mean_ranks.min())
    max_r = float(np.ceil(mean_ranks.max())) + 0.2 * (mean_ranks.max() - mean_ranks.min())
    ax.set_xlim(min_r, max_r); ax.set_ylim(-(k + 4) * 0.4, 1.6); ax.axis("off")

    # CD bar at top
    cd_x0 = min_r + 0.1 * (max_r - min_r); cd_x1 = cd_x0 + cd
    ax.plot([cd_x0, cd_x1], [1.2, 1.2], "k", lw=2)
    ax.plot([cd_x0, cd_x0], [1.1, 1.3], "k", lw=2)
    ax.plot([cd_x1, cd_x1], [1.1, 1.3], "k", lw=2)
    ax.text((cd_x0 + cd_x1) / 2, 1.4, f"CD = {cd:.2f}", ha="center", va="bottom", fontsize=10)

    for r in np.arange(np.ceil(min_r), np.floor(max_r) + 1):
        ax.plot([r, r], [0.7, 0.85], "k", lw=1)
        ax.text(r, 0.95, f"{int(r)}", ha="center", va="bottom", fontsize=9)
    ax.plot([min_r, max_r], [0.7, 0.7], "k", lw=1)

    half = (k + 1) // 2
    left = list(reversed(tuners[:half])); right = tuners[half:]
    def draw_branch(tname, y, side):
        x_rank = mean_ranks[tname]
        ax.plot([x_rank, x_rank], [0.7, y], "k", lw=1)
        if side == "left":
            ax.plot([x_rank, min_r + 0.05 * (max_r - min_r)], [y, y], "k", lw=1)
            ax.text(min_r + 0.04 * (max_r - min_r), y,
                    f"{tname}  ({mean_ranks[tname]:.2f})", ha="right", va="center", fontsize=10)
        else:
            ax.plot([x_rank, max_r - 0.05 * (max_r - min_r)], [y, y], "k", lw=1)
            ax.text(max_r - 0.04 * (max_r - min_r), y,
                    f"({mean_ranks[tname]:.2f})  {tname}", ha="left", va="center", fontsize=10)
    for idx, t in enumerate(left):
        draw_branch(t, 0.2 - idx * 0.5 - 0.3, "left")
    for idx, t in enumerate(right):
        draw_branch(t, 0.2 - idx * 0.5 - 0.3, "right")

    # Clique bars (groups of consecutive tuners within CD of each other)
    cliques = []; i = 0
    while i < k:
        j = i
        while j + 1 < k and mean_ranks[tuners[j + 1]] - mean_ranks[tuners[i]] <= cd:
            j += 1
        if j > i:
            cliques.append(tuners[i:j + 1])
        i += 1
    for ci, clique in enumerate(cliques):
        x0, x1 = mean_ranks[clique[0]], mean_ranks[clique[-1]]
        y = -0.2 - (k + 1) * 0.05 - ci * 0.10
        ax.plot([x0 - 0.03, x1 + 0.03], [y, y], "k", lw=4, solid_capstyle="butt")

    if title: ax.set_title(title, fontsize=11)
    return ax.figure

---
## 10. Demo — synthetic 8-tuner benchmark

Build a small synthetic dataset where we *know* HDGPSO is the best tuner, then run the full Demsar battery on it.

---
## 12. Reading the saved paper benchmark results

Load `summary.csv` from the saved paper benchmark (generated by `benchmarks/run_claim_check_v5.py`) and verify the 6 paper claims. If you haven't run the benchmark yet, this cell will print a notice and skip.

In [ ]:
cd_diagram(demo_summary, alpha=0.05,
           title=f"CD diagram (synthetic n={len(build_rank_matrix(demo_summary))} cells, α=0.05)")
plt.show()

---
## 11. Reading the saved paper benchmark results

Load `summary.csv` from the saved paper benchmark (generated by `benchmarks/run_claim_check_v5.py`) and verify the 6 paper claims. If you haven't run the benchmark yet, this cell will print a notice and skip.

In [ ]:
import os

PATHS = [
    "../results_claim_check_v5/summary.csv",
    "../results_claim_check_v3/summary.csv",   # fallback if v5 hasn't completed yet
]
summary_path = next((p for p in PATHS if os.path.exists(p)), None)

if summary_path is None:
    print("No benchmark CSV found yet. Run:\n  cd benchmarks && python run_claim_check_v5.py\n"
          "  (takes ~3-4 hr)")
    summary = None
else:
    summary = pd.read_csv(summary_path)
    print(f"Loaded {summary_path}: {len(summary)} rows.")
    print(f"Tuners: {sorted(summary['tuner'].unique())}")

In [ ]:
if summary is not None:
    fr = friedman_test(summary, alpha=0.05)
    print(f"CLAIM 1 — Friedman rejects H0 at alpha=0.05: "
          f"{'PASS' if fr['reject_null'] else 'FAIL'} (p={fr['pvalue']:.2e})")

    mean_ranks = build_rank_matrix(summary).mean(axis=0).sort_values()
    print("\nCLAIM 2 — Mean ranks (lower=better):")
    for t, r in mean_ranks.items():
        marker = "  <-- winner" if r == mean_ranks.iloc[0] else ""
        print(f"  {t:14s}  {r:.3f}{marker}")

---
## 13. Conclusion

This notebook contains the full HDGPSO implementation, the multi-fidelity variant, and the Demsar (2006) statistical machinery in one runnable document.

Suggestions for next steps:

- For production use, install the package directly with `pip install git+https://github.com/ashuxen/hdgpso.git` and write a custom search space and objective. The classes available through the package are the same ones implemented above.
- For multi-fidelity workflows, define an objective with signature `def objective(params, fidelity=1.0):` so that lower fidelity values map to cheaper, slightly noisier evaluations. Then use `HDGPSOMF` instead of `HDGPSO`. On the benchmark mix used in the paper, the multi-fidelity variant did not show a consistent advantage; see the experimental notes for context.
- To reproduce the full paper benchmark, see `benchmarks/run_claim_check_v5.py` and `benchmarks/run_budget_sweep.py`.

Bug reports and suggestions are welcome at https://github.com/ashuxen/hdgpso.


In [ ]:
if summary is not None:
    cd_diagram(summary, alpha=0.05,
               title=f"Paper benchmark CD diagram (n={len(build_rank_matrix(summary))} cells)")
    plt.show()

---
## 12. Conclusion

This notebook contains the complete `hdgpso` library implementation along with the Demsar (2006) statistical machinery, in one runnable sequence.

**Where to go next:**

- For production use: install the pip package — `pip install hdgpso` — and write your own search space + objective. The package's `hdgpso.HDGPSO` and `hdgpso.HDGPSOMF` classes are exactly what's defined above.
- For multi-fidelity: write `def objective(params, fidelity=1.0)` with a cheaper / noisier proxy when fidelity < 1, and let HDGPSOMF allocate budget.
- **For reproducing the paper**: run `benchmarks/run_claim_check_v5.py` and `benchmarks/run_budget_sweep.py`, then `benchmarks/_final_report.py` to regenerate every figure in `paper/paper_draft.md`.
- **For modifying the algorithm**: this notebook is the easiest entry point — duplicate the HDGPSO class cell and start hacking. The unit tests in `tests/test_hdgpso.py` will tell you if you broke anything.
- **For citation**: see [CITATION.cff](../CITATION.cff).

References to all the foundational algorithms (Mirjalili 2014 GWO, Storn 1997 DE, Kennedy 1995 PSO, Demsar 2006 statistics, Falkner 2018 BOHB) are listed in `README.md` and `CHANGELOG.md`.